In [1]:
system_prompt = """

you are an Agent on a Digital Skills Websites, what you do is to answer enquiries and questions about the Business Analytics with AI, using thw information provided.

Always respond warmly, if you are unable to answer, refer the user to Ruth on 08000000000
"""

In [2]:
# %% Cell 1: Create project structure
import os

BASE = "rag-agent"

dirs = [
    f"{BASE}/app/rag",
    f"{BASE}/data/uploads",
    f"{BASE}/data/faiss_index",
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

# __init__.py files so app/ and app/rag/ are proper packages
open(f"{BASE}/app/__init__.py", "w").close()
open(f"{BASE}/app/rag/__init__.py", "w").close()

print("Folder structure created:")
for root, subdirs, files in os.walk(BASE):
    level = root.replace(BASE, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

Folder structure created:
rag-agent/
  app/
    rag/
  data/
    faiss_index/
    uploads/


In [3]:
# %% Cell 2: requirements.txt
requirements = """\
fastapi==0.115.0
uvicorn[standard]==0.30.6
python-multipart==0.0.9
groq==0.11.0
faiss-cpu==1.8.0.post1
sentence-transformers==3.1.1
pypdf==4.3.1
numpy==1.26.4
python-dotenv==1.0.1
pydantic==2.9.2
"""

with open("rag-agent/requirements.txt", "w") as f:
    f.write(requirements)

print(requirements)

fastapi==0.115.0
uvicorn[standard]==0.30.6
python-multipart==0.0.9
groq==0.11.0
faiss-cpu==1.8.0.post1
sentence-transformers==3.1.1
pypdf==4.3.1
numpy==1.26.4
python-dotenv==1.0.1
pydantic==2.9.2



In [4]:
# %% Cell 3: Dockerfile
dockerfile = """\
FROM python:3.11-slim

WORKDIR /code

# System deps for building faiss / sentence-transformers wheels smoothly
RUN apt-get update && apt-get install -y --no-install-recommends \\
    build-essential \\
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app ./app
COPY data ./data

# Render/Railway both inject $PORT at runtime
ENV PORT=8000
EXPOSE 8000

CMD ["sh", "-c", "uvicorn app.main:app --host 0.0.0.0 --port ${PORT}"]
"""

with open("rag-agent/Dockerfile", "w") as f:
    f.write(dockerfile)

print(dockerfile)

FROM python:3.11-slim

WORKDIR /code

# System deps for building faiss / sentence-transformers wheels smoothly
RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app ./app
COPY data ./data

# Render/Railway both inject $PORT at runtime
ENV PORT=8000
EXPOSE 8000

CMD ["sh", "-c", "uvicorn app.main:app --host 0.0.0.0 --port ${PORT}"]



In [5]:
# %% Cell 4: .dockerignore + .env
dockerignore = """
__pycache__
*.pyc
.env
.git
.ipynb_checkpoints
data/uploads/*
"""

env_content = """
GROQ_API_KEY=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
GROQ_MODEL=openai/gpt-oss-120b
EMBEDDING_MODEL=BAAI/bge-small-en-v1.5
CHUNK_SIZE=800
CHUNK_OVERLAP=120
TOP_K=4
PORT=8000
"""


with open("rag-agent/.dockerignore", "w") as f:
    f.write(dockerignore)

with open("rag-agent/.env.example", "w") as f:
    f.write(env_content)

print("Written .dockerignore and .env")

Written .dockerignore and .env


In [6]:
# %% Cell 5: Persist your existing system_prompt variable to a file
assert "system_prompt" in dir(), "Define your `system_prompt` variable in an earlier cell first."

with open("rag-agent/app/system_prompt.txt", "w") as f:
    f.write(system_prompt)

print("system_prompt.txt written. Preview:")
print(system_prompt[:300], "...")

system_prompt.txt written. Preview:


you are an Agent on a Digital Skills Websites, what you do is to answer enquiries and questions about the Business Analytics with AI, using thw information provided.

Always respond warmly, if you are unable to answer, refer the user to Ruth on 08000000000
 ...


In [7]:
# %% Cell 6: app/config.py
config_py = '''\
import os
from pathlib import Path

APP_DIR = Path(__file__).resolve().parent
DATA_DIR = APP_DIR.parent / "data"
UPLOAD_DIR = DATA_DIR / "uploads"
INDEX_DIR = DATA_DIR / "faiss_index"
INDEX_PATH = INDEX_DIR / "index.faiss"
METADATA_PATH = INDEX_DIR / "metadata.json"
SYSTEM_PROMPT_PATH = APP_DIR / "system_prompt.txt"

UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise RuntimeError(
        "GROQ_API_KEY is not set. Add it as an environment variable / platform secret."
    )

GROQ_MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-120b")
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "BAAI/bge-small-en-v1.5")
CHUNK_SIZE = int(os.environ.get("CHUNK_SIZE", 800)
CHUNK_OVERLAP = int(os.environ.get("CHUNK_OVERLAP", 120)
TOP_K = int(os.environ.get("TOP_K", 4))

def load_system_prompt() -> str:
    if SYSTEM_PROMPT_PATH.exists():
        return SYSTEM_PROMPT_PATH.read_text()
    return "You are a helpful RAG assistant. Answer using only the provided context."
'''

with open("rag-agent/app/config.py", "w") as f:
    f.write(config_py)

print(config_py)

import os
from pathlib import Path

APP_DIR = Path(__file__).resolve().parent
DATA_DIR = APP_DIR.parent / "data"
UPLOAD_DIR = DATA_DIR / "uploads"
INDEX_DIR = DATA_DIR / "faiss_index"
INDEX_PATH = INDEX_DIR / "index.faiss"
METADATA_PATH = INDEX_DIR / "metadata.json"
SYSTEM_PROMPT_PATH = APP_DIR / "system_prompt.txt"

UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise RuntimeError(
        "GROQ_API_KEY is not set. Add it as an environment variable / platform secret."
    )

GROQ_MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-120b")
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "BAAI/bge-small-en-v1.5")
CHUNK_SIZE = int(os.environ.get("CHUNK_SIZE", 800)
CHUNK_OVERLAP = int(os.environ.get("CHUNK_OVERLAP", 120)
TOP_K = int(os.environ.get("TOP_K", 4))

def load_system_prompt() -> str:
    if SYSTEM_PROMPT_PATH.exists():
        return SYSTEM_PROMPT_PATH.read_tex

In [8]:
# %% Cell 7: app/rag/embeddings.py
embeddings_py = '''\
from sentence_transformers import SentenceTransformer
import numpy as np
from app.config import EMBEDDING_MODEL

_model = None

def get_embedder():
    global _model
    if _model is None:
        _model = SentenceTransformer(EMBEDDING_MODEL)
    return _model

def embed_texts(texts: list[str]) -> np.ndarray:
    """Returns float32 numpy array of shape (n, dim), L2-normalized for cosine similarity via inner product."""
    model = get_embedder()
    vectors = model.encode(texts, convert_to_numpy=True, show_progress_bar=False)
    vectors = vectors.astype("float32")
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1e-9
    return vectors / norms

def embed_query(text: str) -> np.ndarray:
    return embed_texts([text])[0]
'''

with open("rag-agent/app/rag/embeddings.py", "w") as f:
    f.write(embeddings_py)

print(embeddings_py)

from sentence_transformers import SentenceTransformer
import numpy as np
from app.config import EMBEDDING_MODEL

_model = None

def get_embedder():
    global _model
    if _model is None:
        _model = SentenceTransformer(EMBEDDING_MODEL)
    return _model

def embed_texts(texts: list[str]) -> np.ndarray:
    """Returns float32 numpy array of shape (n, dim), L2-normalized for cosine similarity via inner product."""
    model = get_embedder()
    vectors = model.encode(texts, convert_to_numpy=True, show_progress_bar=False)
    vectors = vectors.astype("float32")
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1e-9
    return vectors / norms

def embed_query(text: str) -> np.ndarray:
    return embed_texts([text])[0]



In [9]:
# %% Cell 8: app/rag/document_loader.py
document_loader_py = '''\
from pypdf import PdfReader

def extract_text_from_pdf(path: str) -> str:
    reader = PdfReader(path)
    pages = []
    for page in reader.pages:
        text = page.extract_text() or ""
        pages.append(text)
    return "\\n".join(pages)

def chunk_text(text: str, chunk_size: int = 800, overlap: int = 150) -> list[str]:
    """Simple sliding-window character chunker with overlap."""
    text = " ".join(text.split())  # normalize whitespace
    if not text:
        return []

    chunks = []
    start = 0
    length = len(text)
    while start < length:
        end = min(start + chunk_size, length)
        chunks.append(text[start:end])
        if end == length:
            break
        start = end - overlap
    return chunks
'''

with open("rag-agent/app/rag/document_loader.py", "w") as f:
    f.write(document_loader_py)

print(document_loader_py)

from pypdf import PdfReader

def extract_text_from_pdf(path: str) -> str:
    reader = PdfReader(path)
    pages = []
    for page in reader.pages:
        text = page.extract_text() or ""
        pages.append(text)
    return "\n".join(pages)

def chunk_text(text: str, chunk_size: int = 800, overlap: int = 150) -> list[str]:
    """Simple sliding-window character chunker with overlap."""
    text = " ".join(text.split())  # normalize whitespace
    if not text:
        return []

    chunks = []
    start = 0
    length = len(text)
    while start < length:
        end = min(start + chunk_size, length)
        chunks.append(text[start:end])
        if end == length:
            break
        start = end - overlap
    return chunks



In [10]:
# %% Cell 9: app/rag/vector_store.py
vector_store_py = '''\
import json
import faiss
import numpy as np
from app.config import INDEX_PATH, METADATA_PATH
from app.rag.embeddings import embed_texts

_index = None
_metadata: list[dict] = []  # aligned by faiss internal id order

def _dim() -> int:
    from app.rag.embeddings import get_embedder
    return get_embedder().get_sentence_embedding_dimension()

def _load():
    global _index, _metadata
    if _index is not None:
        return
    if INDEX_PATH.exists() and METADATA_PATH.exists():
        _index = faiss.read_index(str(INDEX_PATH))
        with open(METADATA_PATH, "r") as f:
            _metadata = json.load(f)
    else:
        _index = faiss.IndexFlatIP(_dim())  # inner product on normalized vecs = cosine sim
        _metadata = []

def _save():
    faiss.write_index(_index, str(INDEX_PATH))
    with open(METADATA_PATH, "w") as f:
        json.dump(_metadata, f)

def add_document(source_name: str, chunks: list[str]) -> int:
    """Embeds and adds chunks to the index. Returns number of chunks added."""
    _load()
    if not chunks:
        return 0
    vectors = embed_texts(chunks)
    _index.add(vectors)
    for chunk in chunks:
        _metadata.append({"source": source_name, "text": chunk})
    _save()
    return len(chunks)

def search(query_vector: np.ndarray, top_k: int = 4) -> list[dict]:
    _load()
    if _index.ntotal == 0:
        return []
    query_vector = query_vector.reshape(1, -1)
    scores, ids = _index.search(query_vector, min(top_k, _index.ntotal))
    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue
        item = _metadata[idx]
        results.append({"text": item["text"], "source": item["source"], "score": float(score)})
    return results

def stats() -> dict:
    _load()
    sources = sorted(set(m["source"] for m in _metadata))
    return {"total_chunks": _index.ntotal, "sources": sources}
'''

with open("rag-agent/app/rag/vector_store.py", "w") as f:
    f.write(vector_store_py)

print(vector_store_py)

import json
import faiss
import numpy as np
from app.config import INDEX_PATH, METADATA_PATH
from app.rag.embeddings import embed_texts

_index = None
_metadata: list[dict] = []  # aligned by faiss internal id order

def _dim() -> int:
    from app.rag.embeddings import get_embedder
    return get_embedder().get_sentence_embedding_dimension()

def _load():
    global _index, _metadata
    if _index is not None:
        return
    if INDEX_PATH.exists() and METADATA_PATH.exists():
        _index = faiss.read_index(str(INDEX_PATH))
        with open(METADATA_PATH, "r") as f:
            _metadata = json.load(f)
    else:
        _index = faiss.IndexFlatIP(_dim())  # inner product on normalized vecs = cosine sim
        _metadata = []

def _save():
    faiss.write_index(_index, str(INDEX_PATH))
    with open(METADATA_PATH, "w") as f:
        json.dump(_metadata, f)

def add_document(source_name: str, chunks: list[str]) -> int:
    """Embeds and adds chunks to the index. Returns number of 

In [11]:
# %% Cell 10: app/rag/agent.py
agent_py = '''\
from groq import Groq
from app.config import GROQ_API_KEY, GROQ_MODEL, TOP_K, load_system_prompt
from app.rag.embeddings import embed_query
from app.rag.vector_store import search

_client = Groq(api_key=GROQ_API_KEY)

def build_context(chunks: list[dict]) -> str:
    if not chunks:
        return "No relevant context was found in the knowledge base."
    parts = []
    for i, c in enumerate(chunks, start=1):
        parts.append(f"[{i}] (source: {c['source']}, relevance: {c['score']:.2f})\\n{c['text']}")
    return "\\n\\n".join(parts)

def answer_query(query: str, top_k: int | None = None) -> dict:
    k = top_k or TOP_K
    query_vec = embed_query(query)
    retrieved = search(query_vec, top_k=k)
    context = build_context(retrieved)

    system_prompt = load_system_prompt()
    user_message = (
        f"Context from knowledge base:\\n{context}\\n\\n"
        f"User question: {query}\\n\\n"
        "Answer using only the context above. If the context doesn't contain "
        "the answer, say so clearly instead of guessing."
    )

    completion = _client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
        ],
    )

    answer = completion.choices[0].message.content
    return {
        "answer": answer,
        "sources": [{"source": c["source"], "score": c["score"]} for c in retrieved],
    }
'''

with open("rag-agent/app/rag/agent.py", "w") as f:
    f.write(agent_py)

print(agent_py)

from groq import Groq
from app.config import GROQ_API_KEY, GROQ_MODEL, TOP_K, load_system_prompt
from app.rag.embeddings import embed_query
from app.rag.vector_store import search

_client = Groq(api_key=GROQ_API_KEY)

def build_context(chunks: list[dict]) -> str:
    if not chunks:
        return "No relevant context was found in the knowledge base."
    parts = []
    for i, c in enumerate(chunks, start=1):
        parts.append(f"[{i}] (source: {c['source']}, relevance: {c['score']:.2f})\n{c['text']}")
    return "\n\n".join(parts)

def answer_query(query: str, top_k: int | None = None) -> dict:
    k = top_k or TOP_K
    query_vec = embed_query(query)
    retrieved = search(query_vec, top_k=k)
    context = build_context(retrieved)

    system_prompt = load_system_prompt()
    user_message = (
        f"Context from knowledge base:\n{context}\n\n"
        f"User question: {query}\n\n"
        "Answer using only the context above. If the context doesn't contain "
        "the answer,

In [12]:
# %% Cell 11: app/main.py
main_py = '''\
import shutil
import uuid
from pathlib import Path

from fastapi import FastAPI, UploadFile, File, HTTPException
from pydantic import BaseModel

from app.config import UPLOAD_DIR
from app.rag.document_loader import extract_text_from_pdf, chunk_text
from app.rag.vector_store import add_document, stats
from app.rag.agent import answer_query

app = FastAPI(title="RAG Agent API")


class ChatRequest(BaseModel):
    query: str
    top_k: int | None = None


class ChatResponse(BaseModel):
    answer: str
    sources: list[dict]


@app.get("/health")
def health():
    return {"status": "ok", **stats()}


@app.post("/chat", response_model=ChatResponse)
def chat(req: ChatRequest):
    if not req.query.strip():
        raise HTTPException(status_code=400, detail="query must not be empty")
    result = answer_query(req.query, top_k=req.top_k)
    return result


@app.post("/upload")
async def upload_pdf(file: UploadFile = File(...)):
    if not file.filename.lower().endswith(".pdf"):
        raise HTTPException(status_code=400, detail="Only PDF files are supported.")

    safe_name = f"{uuid.uuid4().hex}_{file.filename}"
    dest_path = Path(UPLOAD_DIR) / safe_name

    with open(dest_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    text = extract_text_from_pdf(str(dest_path))
    if not text.strip():
        raise HTTPException(status_code=422, detail="Could not extract any text from this PDF.")

    chunks = chunk_text(text)
    added = add_document(source_name=file.filename, chunks=chunks)

    return {
        "message": f"Uploaded and indexed '{file.filename}'.",
        "chunks_added": added,
        "knowledge_base_stats": stats(),
    }
'''

with open("rag-agent/app/main.py", "w") as f:
    f.write(main_py)

print(main_py)

import shutil
import uuid
from pathlib import Path

from fastapi import FastAPI, UploadFile, File, HTTPException
from pydantic import BaseModel

from app.config import UPLOAD_DIR
from app.rag.document_loader import extract_text_from_pdf, chunk_text
from app.rag.vector_store import add_document, stats
from app.rag.agent import answer_query

app = FastAPI(title="RAG Agent API")


class ChatRequest(BaseModel):
    query: str
    top_k: int | None = None


class ChatResponse(BaseModel):
    answer: str
    sources: list[dict]


@app.get("/health")
def health():
    return {"status": "ok", **stats()}


@app.post("/chat", response_model=ChatResponse)
def chat(req: ChatRequest):
    if not req.query.strip():
        raise HTTPException(status_code=400, detail="query must not be empty")
    result = answer_query(req.query, top_k=req.top_k)
    return result


@app.post("/upload")
async def upload_pdf(file: UploadFile = File(...)):
    if not file.filename.lower().endswith(".pdf"):
        rais

In [13]:
# %% Cell 12: README.md
readme = """\
# RAG Agent (FastAPI + FAISS-CPU + Groq)

## Endpoints
- GET  /health         -> index stats
- POST /chat           -> {"query": "..."} -> {"answer": "...", "sources": [...]}
- POST /upload         -> multipart form file=<pdf>

## Deploy on Render / Railway
1. Push this folder to a GitHub repo.
2. Create a new Web Service, point it at the repo, choose "Docker" as the environment.
3. Set env var GROQ_API_KEY (and optionally GROQ_MODEL, EMBEDDING_MODEL, TOP_K).
4. Both platforms inject $PORT automatically; the Dockerfile CMD already uses it.
5. IMPORTANT: default disk is ephemeral. Attach a persistent volume mounted at
   /code/data if you need the FAISS index to survive redeploys.
"""

with open("rag-agent/README.md", "w") as f:
    f.write(readme)

print(readme)

# RAG Agent (FastAPI + FAISS-CPU + Groq)

## Endpoints
- GET  /health         -> index stats
- POST /chat           -> {"query": "..."} -> {"answer": "...", "sources": [...]}
- POST /upload         -> multipart form file=<pdf>

## Deploy on Render / Railway
1. Push this folder to a GitHub repo.
2. Create a new Web Service, point it at the repo, choose "Docker" as the environment.
3. Set env var GROQ_API_KEY (and optionally GROQ_MODEL, EMBEDDING_MODEL, TOP_K).
4. Both platforms inject $PORT automatically; the Dockerfile CMD already uses it.
5. IMPORTANT: default disk is ephemeral. Attach a persistent volume mounted at
   /code/data if you need the FAISS index to survive redeploys.



In [14]:
# %% Cell 13: Zip the project
import shutil

shutil.make_archive("rag-agent", "zip", "rag-agent")
print("Created rag-agent.zip — download it from the Colab file browser,")
print("unzip locally, git init, and push to GitHub for Render/Railway to build from.")

from google.colab import files
files.download("rag-agent.zip")

Created rag-agent.zip — download it from the Colab file browser,
unzip locally, git init, and push to GitHub for Render/Railway to build from.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>